In [1]:
# ================================================================
# Notebook 12:
# CWQ Frontier Selection and State Generation
#
# Runtime: CPU
# Prerequisite: completed Notebook 11
#
# No downloads or model training.
# No test search policies or test states are generated.
# Existing WebQSP artifacts remain unchanged.
# ================================================================

import collections
import gzip
import hashlib
import json
import random
import re
import time
from pathlib import Path

# ----------------------------
# Configuration
# ----------------------------

ROOT = Path("/content/drive/MyDrive/SetStop_KGQA")

MAX_HOPS = 4
FRONTIER_LIMITS = [16, 32, 64, 128]
ACTION_CAP = 24
RECALL_TOLERANCE = 0.02

# Save these intermediate steps, plus every trajectory's final state.
SNAPSHOT_STEPS = [0, 2, 4, 8, 12, 18, 24]

# STORAGE sampling only.
# This does not reduce the selected search frontier limit.
TRAIN_CANDIDATES_PER_STATE = 16

# Save completed chunks to support automatic resumption.
CHUNK_QUESTIONS = 100

SEED = 42
CODE_VERSION = "cwq_states_v1_0"


# ----------------------------
# File and reproducibility helpers
# ----------------------------

def fingerprint(value):
    text = json.dumps(value, sort_keys=True)
    return hashlib.sha256(text.encode()).hexdigest()


def file_hash(path):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for block in iter(
            lambda: file.read(8 * 1024**2),
            b"",
        ):
            digest.update(block)
    return digest.hexdigest()


def save_json(path, value):
    temporary = path.with_name(path.name + ".partial")

    with open(temporary, "w", encoding="utf-8") as file:
        json.dump(
            value,
            file,
            indent=2,
            ensure_ascii=False,
        )

    temporary.replace(path)


def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))


def rows(path):
    with open(path, encoding="utf-8") as file:
        for line_number, text in enumerate(file, 1):
            if not text.strip():
                continue

            value = json.loads(text)

            if not isinstance(value, dict):
                raise ValueError(
                    f"Invalid JSON object at {path}:{line_number}"
                )

            yield line_number, value


def stable_seed(*values):
    return int(
        fingerprint([SEED, *values])[:16],
        16,
    )


# ----------------------------
# Dataset parsing
# ----------------------------

def entity_id(value):
    if isinstance(value, dict):
        value = value.get(
            "kb_id",
            value.get("id", value.get("entity")),
        )

    if isinstance(value, int) and not isinstance(value, bool):
        return value if 0 <= value < len(ENTITIES) else None

    if isinstance(value, str):
        return ENTITY_TO_ID.get(value)

    return None


def relation_id(value):
    if isinstance(value, int) and not isinstance(value, bool):
        return value if 0 <= value < len(RELATIONS) else None

    if isinstance(value, str):
        return RELATION_TO_ID.get(value)

    return None


def question_id(row):
    value = row.get("id", row.get("qid"))

    if value is None or not str(value).strip():
        raise ValueError("Missing question ID")

    return str(value)


def parse_example(row):
    question = row.get("question")
    subgraph = row.get("subgraph")

    if (
        not isinstance(question, str)
        or not question.strip()
        or not isinstance(subgraph, dict)
    ):
        raise ValueError(
            f"Invalid question/subgraph: {question_id(row)}"
        )

    nodes = {
        entity_id(value)
        for value in subgraph["entities"]
    }

    edge_list = subgraph.get(
        "tuples",
        subgraph.get("triples"),
    )

    if not isinstance(edge_list, list):
        raise ValueError(
            f"Missing edge list: {question_id(row)}"
        )

    edges = set()

    for edge in edge_list:
        if (
            not isinstance(edge, (list, tuple))
            or len(edge) != 3
        ):
            raise ValueError(
                f"Invalid edge: {question_id(row)}"
            )

        source = entity_id(edge[0])
        relation = relation_id(edge[1])
        target = entity_id(edge[2])

        if source is None or relation is None or target is None:
            raise ValueError(
                f"Unmapped graph edge: {question_id(row)}"
            )

        edges.add((source, relation, target))

    nodes |= {
        node
        for source, relation, target in edges
        for node in (source, target)
    }

    gold = {
        entity_id(value)
        for value in row["answers"]
    }

    topics = {
        entity_id(value)
        for value in row.get(
            "entities",
            row.get("topic_entities", []),
        )
    }

    if None in nodes or None in gold or None in topics:
        raise ValueError(
            "Unmapped entity in selected question: "
            f"{question_id(row)}"
        )

    eligible = (
        2 <= len(gold) <= 50
        and gold <= nodes
        and bool(topics)
        and topics <= nodes
    )

    if not eligible:
        raise ValueError(
            "Selected question no longer meets eligibility: "
            f"{question_id(row)}"
        )

    adjacency = collections.defaultdict(
        lambda: collections.defaultdict(set)
    )
    reverse_adjacency = collections.defaultdict(list)

    # Preserve the published edge directions.
    for source, relation, target in sorted(edges):
        adjacency[source][relation].add(target)
        reverse_adjacency[target].append((source, relation))

    adjacency = {
        source: {
            relation: sorted(targets)
            for relation, targets in group.items()
        }
        for source, group in adjacency.items()
    }

    return {
        "qid": question_id(row),
        "question": question.strip(),
        "gold": gold,
        "topics": topics,
        "adj": adjacency,
        "reverse": reverse_adjacency,
    }


# ----------------------------
# Frontier and reachability
# ----------------------------

def tokens(text):
    return set(
        re.findall(
            r"[a-z0-9]+",
            text.lower().replace("_", " "),
        )
    )


def get_frontier(example, depth, expanded):
    branches = [
        (source, relation, depth[source], targets)
        for source in sorted(depth)
        if depth[source] < MAX_HOPS
        for relation, targets in sorted(
            example["adj"].get(source, {}).items()
        )
        if (source, relation) not in expanded
    ]

    question_tokens = tokens(example["question"])

    return sorted(
        branches,
        key=lambda branch: (
            -len(
                question_tokens
                & RELATION_TOKENS[branch[1]]
            ),
            branch[2],
            -len(branch[3]),
            branch[1],
            branch[0],
        ),
    )


def reachability_levels(example, unseen_gold, expanded):
    """
    levels[h][node] is a bitset of unseen gold entities reachable
    within at most h further directed, unexpanded hops.

    This is offline supervision/diagnostic computation.
    These bitsets and gold IDs never enter model_text.
    """
    current = {
        gold_entity: 1 << index
        for index, gold_entity in enumerate(sorted(unseen_gold))
    }

    levels = [current]

    for _ in range(MAX_HOPS - 1):
        following = dict(current)

        for target, mask in current.items():
            for source, relation in example["reverse"].get(
                target,
                [],
            ):
                if (source, relation) not in expanded:
                    following[source] = (
                        following.get(source, 0) | mask
                    )

        current = following
        levels.append(current)

    return levels


def branch_gain(branch, levels):
    source, relation, source_depth, targets = branch
    remaining_hops = MAX_HOPS - source_depth - 1

    mask = 0

    for target in targets:
        mask |= levels[remaining_hops].get(target, 0)

    return mask.bit_count()


def reference_bfs_gain(
    example,
    branch,
    unseen_gold,
    expanded,
):
    """
    Independent forward-BFS implementation used to check
    the gain labels generated by reverse dynamic programming.
    """
    source, relation, source_depth, targets = branch

    depth = {
        target: source_depth + 1
        for target in targets
    }
    queue = collections.deque(sorted(targets))

    while queue:
        node = queue.popleft()

        if depth[node] >= MAX_HOPS:
            continue

        for next_relation, children in example["adj"].get(
            node,
            {},
        ).items():
            if (node, next_relation) in expanded:
                continue

            for child in children:
                if child not in depth:
                    depth[child] = depth[node] + 1
                    queue.append(child)

    return len(unseen_gold.intersection(depth))


def expand_branch(
    branch,
    depth,
    expanded,
    found,
    example,
):
    source, relation, source_depth, targets = branch
    expanded.add((source, relation))

    for target in targets:
        depth[target] = min(
            depth.get(target, MAX_HOPS + 1),
            source_depth + 1,
        )

    found.update(example["gold"].intersection(targets))


# ----------------------------
# Development frontier comparison
# ----------------------------

def oracle_search(example, frontier_limit):
    depth = {
        topic: 0
        for topic in example["topics"]
    }
    found = example["gold"].intersection(depth)
    expanded = set()

    actions = 0
    retrieved_edges = 0
    stop_reason = "action_cap"

    while actions < ACTION_CAP:
        full_frontier = get_frontier(
            example,
            depth,
            expanded,
        )

        pool = (
            full_frontier
            if frontier_limit is None
            else full_frontier[:frontier_limit]
        )

        if not pool:
            stop_reason = "empty_frontier"
            break

        levels = reachability_levels(
            example,
            example["gold"] - found,
            expanded,
        )

        gains = [
            branch_gain(branch, levels)
            for branch in pool
        ]

        best_index = max(
            range(len(pool)),
            key=lambda index: gains[index],
        )

        if gains[best_index] == 0:
            stop_reason = "oracle_shortlist_zero_gain"
            break

        chosen = pool[best_index]

        expand_branch(
            chosen,
            depth,
            expanded,
            found,
            example,
        )

        retrieved_edges += len(chosen[3])
        actions += 1

    recall = len(found) / len(example["gold"])

    return {
        "qid": example["qid"],
        "frontier_limit": frontier_limit,
        "recall": recall,
        "oracle_recognition_f1": 2 * recall / (1 + recall),
        "exact": int(found == example["gold"]),
        "actions": actions,
        "retrieved_edges": retrieved_edges,
        "stop_reason": stop_reason,
    }


# ----------------------------
# Controller input and labels
# ----------------------------

def controller_text(
    question,
    found_count,
    visited_count,
    step,
    branch,
    history,
):
    # Same input layout used in the WebQSP controller.
    history_text = (
        " -> ".join(history[-6:])
        if history
        else "none"
    )

    return (
        f"Question: {question}\n"
        f"Answers found so far: {found_count}\n"
        f"Visited entities: {visited_count}\n"
        f"Search step: {step}\n"
        f"Source depth: {branch[2]}\n"
        f"Candidate relation: {RELATIONS[branch[1]]}\n"
        f"Candidate target count: {len(branch[3])}\n"
        f"Recent relations: {history_text}"
    )


def gain_class(value):
    return "3_plus" if value >= 3 else str(value)


def generate_trajectory(
    example,
    policy,
    split,
    frontier_limit,
    checks,
):
    depth = {
        topic: 0
        for topic in example["topics"]
    }
    found = example["gold"].intersection(depth)

    expanded = set()
    history = []

    trajectory_rng = random.Random(
        stable_seed(
            "trajectory",
            example["qid"],
            policy,
        )
    )

    for step in range(ACTION_CAP + 1):
        full_frontier = get_frontier(
            example,
            depth,
            expanded,
        )
        pool = full_frontier[:frontier_limit]
        unseen_gold = example["gold"] - found

        snapshot = step in SNAPSHOT_STEPS

        levels = None
        pool_gains = None

        if (
            policy == "oracle"
            or snapshot
            or step == ACTION_CAP
            or not pool
        ):
            levels = reachability_levels(
                example,
                unseen_gold,
                expanded,
            )
            pool_gains = [
                branch_gain(branch, levels)
                for branch in pool
            ]

        terminal_reason = None

        if step == ACTION_CAP:
            terminal_reason = "action_cap"

        elif not pool:
            terminal_reason = "empty_frontier"

        elif (
            policy == "oracle"
            and max(pool_gains, default=0) == 0
        ):
            terminal_reason = "oracle_shortlist_zero_gain"

        if snapshot or terminal_reason is not None:
            if levels is None:
                levels = reachability_levels(
                    example,
                    unseen_gold,
                    expanded,
                )
                pool_gains = [
                    branch_gain(branch, levels)
                    for branch in pool
                ]

            full_gains = [
                branch_gain(branch, levels)
                for branch in full_frontier
            ]

            full_positive_count = sum(
                value > 0
                for value in full_gains
            )
            shortlist_positive_count = sum(
                value > 0
                for value in pool_gains
            )

            indices = list(range(len(pool)))

            # Training storage sampling is independent of gain labels.
            # Development retains all shortlisted candidates.
            if (
                split == "train"
                and len(indices) > TRAIN_CANDIDATES_PER_STATE
            ):
                storage_rng = random.Random(
                    stable_seed(
                        "storage",
                        example["qid"],
                        policy,
                        step,
                    )
                )

                indices = sorted(
                    storage_rng.sample(
                        indices,
                        TRAIN_CANDIDATES_PER_STATE,
                    )
                )

            candidates = []

            for index in indices:
                branch = pool[index]
                value = pool_gains[index]

                candidates.append({
                    "source": branch[0],
                    "relation": branch[1],
                    "source_depth": branch[2],
                    "target_count": len(branch[3]),
                    "relation_text": RELATIONS[branch[1]],
                    "gain": value,
                    "gain_class": gain_class(value),
                    "model_text": controller_text(
                        example["question"],
                        len(found),
                        len(depth),
                        step,
                        branch,
                        history,
                    ),
                })

            # Check the first and last stored actions in selected states
            # against the independent forward-BFS implementation.
            if checks["frames_left"] > 0:
                checked_indices = sorted(
                    set(indices[:1] + indices[-1:])
                )

                for index in checked_indices:
                    reference_gain = reference_bfs_gain(
                        example,
                        pool[index],
                        unseen_gold,
                        expanded,
                    )

                    if reference_gain != pool_gains[index]:
                        raise AssertionError(
                            "BFS gain mismatch: "
                            f"{example['qid']}, step={step}"
                        )

                    checks["passed"] += 1

                checks["frames_left"] -= 1

            yield {
                "state_id": (
                    f"{example['qid']}::{policy}::{step}"
                ),
                "split": split,
                "qid": example["qid"],
                "question": example["question"],
                "trajectory_policy": policy,
                "step": step,
                "found_count": len(found),
                "visited_count": len(depth),
                "frontier_count": len(full_frontier),
                "shortlist_candidate_count": len(pool),
                "candidate_count": len(candidates),
                "full_positive_count": full_positive_count,
                "shortlist_positive_count": shortlist_positive_count,

                # Global label: use the FULL frontier.
                "stop_label": int(full_positive_count == 0),

                # Separate diagnostic label for the shortlist.
                "shortlist_stop_label": int(
                    shortlist_positive_count == 0
                ),

                "shortlist_hides_all_positive": int(
                    full_positive_count > 0
                    and shortlist_positive_count == 0
                ),

                "candidates_sampled": len(indices) < len(pool),

                # Target/diagnostic metadata, never model input.
                "all_answers_found": int(not unseen_gold),
                "remaining_gold_count": len(unseen_gold),

                "history_relations": history[-6:],
                "terminal": terminal_reason is not None,
                "terminal_reason": terminal_reason,
                "candidates": candidates,
            }

        if terminal_reason is not None:
            return

        if policy == "oracle":
            chosen_index = max(
                range(len(pool)),
                key=lambda index: pool_gains[index],
            )
            chosen = pool[chosen_index]

        elif policy == "relation":
            chosen = pool[0]

        elif policy == "random":
            chosen = trajectory_rng.choice(pool)

        else:
            raise ValueError(f"Unknown policy: {policy}")

        expand_branch(
            chosen,
            depth,
            expanded,
            found,
            example,
        )
        history.append(RELATIONS[chosen[1]])


# ----------------------------
# State integrity and input audit
# ----------------------------

def audit_frame(row, split, eligible_ids):
    if (
        row["split"] != split
        or row["qid"] not in eligible_ids
    ):
        raise AssertionError(
            "Wrong split/question in state file"
        )

    if row["candidate_count"] != len(row["candidates"]):
        raise AssertionError("Candidate count mismatch")

    if not (
        0 <= row["found_count"] <= row["visited_count"]
    ):
        raise AssertionError("Invalid observed counts")

    if row["stop_label"] != int(
        row["full_positive_count"] == 0
    ):
        raise AssertionError(
            "Incorrect full-frontier STOP label"
        )

    if row["shortlist_stop_label"] != int(
        row["shortlist_positive_count"] == 0
    ):
        raise AssertionError(
            "Incorrect shortlist STOP label"
        )

    expected_hidden = int(
        row["full_positive_count"] > 0
        and row["shortlist_positive_count"] == 0
    )

    if row["shortlist_hides_all_positive"] != expected_hidden:
        raise AssertionError(
            "Incorrect shortlist-visibility flag"
        )

    if not row["candidates_sampled"]:
        if (
            row["candidate_count"]
            != row["shortlist_candidate_count"]
        ):
            raise AssertionError(
                "Missing unsampled candidates"
            )

        stored_positive_count = sum(
            candidate["gain"] > 0
            for candidate in row["candidates"]
        )

        if (
            stored_positive_count
            != row["shortlist_positive_count"]
        ):
            raise AssertionError(
                "Shortlist gain count mismatch"
            )

    if split == "dev" and row["candidates_sampled"]:
        raise AssertionError(
            "Development candidates must not be sampled"
        )

    for candidate in row["candidates"]:
        if (
            candidate["gain"] < 0
            or candidate["gain_class"]
            != gain_class(candidate["gain"])
        ):
            raise AssertionError(
                "Incorrect gain class"
            )

        # Rebuild the entire model input using only approved features.
        # Do not search for words such as "gold" inside the question.
        branch = (
            candidate["source"],
            candidate["relation"],
            candidate["source_depth"],
            range(candidate["target_count"]),
        )

        expected_text = controller_text(
            row["question"],
            row["found_count"],
            row["visited_count"],
            row["step"],
            branch,
            row["history_relations"],
        )

        if candidate["model_text"] != expected_text:
            raise AssertionError(
                "Model text contains unapproved/mismatched fields"
            )


# ----------------------------
# Resumable split generation
# ----------------------------

def build_split(
    split,
    selected_rows,
    data_dir,
    output_dir,
    processed_dir,
    signature,
    frontier_limit,
):
    selected_ids = [
        row["qid"]
        for row in selected_rows
    ]
    eligible_ids = set(selected_ids)

    if not selected_ids:
        raise RuntimeError(
            f"No selected {split} questions"
        )

    chunks = [
        selected_ids[index:index + CHUNK_QUESTIONS]
        for index in range(
            0,
            len(selected_ids),
            CHUNK_QUESTIONS,
        )
    ]

    shard_dir = (
        output_dir
        / "state_shards"
        / signature[:12]
        / split
    )
    shard_dir.mkdir(parents=True, exist_ok=True)

    shard_paths = []
    metadata_paths = []
    complete_chunks = set()

    for index, members in enumerate(chunks):
        shard_path = (
            shard_dir / f"chunk_{index:04d}.jsonl.gz"
        )
        metadata_path = (
            shard_dir / f"chunk_{index:04d}.json"
        )

        shard_paths.append(shard_path)
        metadata_paths.append(metadata_path)

        metadata = (
            read_json(metadata_path)
            if shard_path.exists() and metadata_path.exists()
            else None
        )

        valid_cache = (
            metadata is not None
            and metadata.get("signature") == signature
            and metadata.get("question_ids") == members
            and metadata.get("bytes")
            == shard_path.stat().st_size
            and metadata.get("sha256")
            == file_hash(shard_path)
        )

        if valid_cache:
            complete_chunks.add(index)

    print(
        f"{split}: {len(selected_ids):,} questions; "
        f"{len(complete_chunks)}/{len(chunks)} cached chunks."
    )

    if len(complete_chunks) < len(chunks):
        pending = {
            qid: index
            for index, members in enumerate(chunks)
            if index not in complete_chunks
            for qid in members
        }

        active_index = None
        handle = None
        temporary_path = None
        checks = None
        written_qids = []

        started = time.time()

        try:
            for _, raw_row in rows(
                data_dir / f"{split}_simple.json"
            ):
                current_qid = question_id(raw_row)

                if current_qid not in pending:
                    continue

                chunk_index = pending[current_qid]

                if active_index != chunk_index:
                    if handle is not None:
                        raise AssertionError(
                            "Manifest order differs from dataset order"
                        )

                    active_index = chunk_index

                    temporary_path = shard_paths[
                        chunk_index
                    ].with_name(
                        shard_paths[chunk_index].name + ".partial"
                    )

                    handle = gzip.open(
                        temporary_path,
                        "wt",
                        encoding="utf-8",
                        compresslevel=3,
                    )

                    written_qids = []
                    checks = {
                        "frames_left": 20,
                        "passed": 0,
                    }

                example = parse_example(raw_row)

                for policy in ["oracle", "relation", "random"]:
                    for frame in generate_trajectory(
                        example,
                        policy,
                        split,
                        frontier_limit,
                        checks,
                    ):
                        audit_frame(
                            frame,
                            split,
                            eligible_ids,
                        )

                        handle.write(
                            json.dumps(
                                frame,
                                ensure_ascii=False,
                            )
                            + "\n"
                        )

                written_qids.append(current_qid)

                if len(written_qids) == len(chunks[chunk_index]):
                    handle.close()
                    handle = None

                    if written_qids != chunks[chunk_index]:
                        raise AssertionError(
                            "Chunk question sequence mismatch"
                        )

                    temporary_path.replace(
                        shard_paths[chunk_index]
                    )

                    save_json(
                        metadata_paths[chunk_index],
                        {
                            "signature": signature,
                            "question_ids": written_qids,
                            "independent_gain_checks": checks["passed"],
                            "bytes": shard_paths[
                                chunk_index
                            ].stat().st_size,
                            "sha256": file_hash(
                                shard_paths[chunk_index]
                            ),
                        },
                    )

                    complete_chunks.add(chunk_index)
                    active_index = None

                    completed_questions = sum(
                        len(chunks[index])
                        for index in complete_chunks
                    )

                    print(
                        f"{split}: "
                        f"{completed_questions:,}/{len(selected_ids):,} "
                        "questions complete | "
                        f"this run {(time.time() - started) / 60:.1f} min",
                        flush=True,
                    )

            if (
                handle is not None
                or len(complete_chunks) != len(chunks)
            ):
                raise RuntimeError(
                    "Some selected questions were not found "
                    "in the raw split"
                )

        finally:
            if handle is not None:
                handle.close()

    # Merge completed chunks into a plain JSONL training file.
    # Audit the persisted records again while merging.
    target_path = (
        processed_dir
        / f"cwq_{split}_states_v1_full.jsonl"
    )
    temporary_path = target_path.with_name(
        target_path.name + ".partial"
    )

    counts = collections.Counter({
        "states": 0,
        "candidates": 0,
        "stop_states": 0,
        "continue_states": 0,
        "shortlist_hidden_states": 0,
        "sampled_states": 0,
        "terminal_states": 0,
    })

    class_counts = collections.Counter()
    policy_counts = collections.Counter()

    seen_states = set()
    seen_questions = set()
    digest = hashlib.sha256()

    with open(temporary_path, "wb") as destination:
        for shard_path in shard_paths:
            with gzip.open(shard_path, "rb") as source:
                for line in source:
                    row = json.loads(line)

                    audit_frame(
                        row,
                        split,
                        eligible_ids,
                    )

                    if row["state_id"] in seen_states:
                        raise AssertionError(
                            "Duplicate state ID"
                        )

                    seen_states.add(row["state_id"])
                    seen_questions.add(row["qid"])

                    counts["states"] += 1
                    counts["candidates"] += row["candidate_count"]
                    counts["stop_states"] += row["stop_label"]
                    counts["continue_states"] += 1 - row["stop_label"]

                    counts["shortlist_hidden_states"] += (
                        row["shortlist_hides_all_positive"]
                    )

                    counts["sampled_states"] += int(
                        row["candidates_sampled"]
                    )
                    counts["terminal_states"] += int(
                        row["terminal"]
                    )

                    policy_counts[row["trajectory_policy"]] += 1

                    class_counts.update(
                        candidate["gain_class"]
                        for candidate in row["candidates"]
                    )

                    destination.write(line)
                    digest.update(line)

    if seen_questions != eligible_ids:
        raise AssertionError(
            "Merged question coverage mismatch"
        )

    temporary_path.replace(target_path)

    independent_checks = sum(
        read_json(path)["independent_gain_checks"]
        for path in metadata_paths
    )

    report = {
        **dict(counts),
        "questions": len(selected_ids),
        "gain_classes": dict(class_counts),
        "trajectory_policies": dict(policy_counts),
        "independent_gain_checks": independent_checks,
        "explicit_input_audit_passed": True,
        "integrity_passed": True,
        "path": str(target_path),
        "bytes": target_path.stat().st_size,
        "sha256": digest.hexdigest(),
    }

    print(split, json.dumps(report, indent=2))

    return report


# ----------------------------
# Main workflow
# ----------------------------

def main():
    from google.colab import drive
    drive.mount("/content/drive")

    started = time.time()

    audit_dir = ROOT / "outputs/cwq_audit_v1"
    data_dir = ROOT / "data/extracted/cwq"
    output_dir = ROOT / "outputs/cwq_preprocessing_v1"
    processed_dir = ROOT / "data/processed"

    output_dir.mkdir(parents=True, exist_ok=True)
    processed_dir.mkdir(parents=True, exist_ok=True)

    required_audit_files = [
        "cwq_dataset_audit_summary_v1.json",
        "cwq_manifest_v1.json",
        "cwq_dev_pilot_baselines_v1.json",
        "artifact_inventory_v1.json",
    ]

    for filename in required_audit_files:
        if not (audit_dir / filename).is_file():
            raise FileNotFoundError(
                f"Missing Notebook-11 output: {audit_dir / filename}"
            )

    summary = read_json(
        audit_dir / "cwq_dataset_audit_summary_v1.json"
    )
    manifest = read_json(
        audit_dir / "cwq_manifest_v1.json"
    )
    pilot = read_json(
        audit_dir / "cwq_dev_pilot_baselines_v1.json"
    )
    inventory = read_json(
        audit_dir / "artifact_inventory_v1.json"
    )

    print("=" * 90)
    print("NOTEBOOK 12: CWQ FRONTIER SELECTION AND STATE GENERATION")
    print("=" * 90)
    print("Runtime: CPU")
    print("No downloads, model training, or test search evaluation.")
    print("Output directory:", output_dir)
    print("Action cap:", ACTION_CAP)
    print("Frontier limits:", FRONTIER_LIMITS)
    print("=" * 90)

    signatures_match = (
        summary["signature"]
        == manifest["signature"]
        == pilot["signature"]
        == inventory["signature"]
    )

    if not signatures_match:
        raise RuntimeError(
            "Notebook-11 artifact signatures disagree"
        )

    # Verify the prior reports and manifest against their inventory.
    for filename in [
        "cwq_dataset_audit_summary_v1.json",
        "cwq_manifest_v1.json",
        "cwq_dev_pilot_baselines_v1.json",
    ]:
        entry = next(
            (
                item
                for item in inventory["files"]
                if item["name"] == filename
            ),
            None,
        )

        path = audit_dir / filename

        if (
            entry is None
            or path.stat().st_size != entry["bytes"]
            or file_hash(path) != entry["sha256"]
        ):
            raise RuntimeError(
                f"Notebook-11 artifact hash mismatch: {filename}"
            )

    if summary["configuration"]["max_hops"] != MAX_HOPS:
        raise RuntimeError(
            "MAX_HOPS must match Notebook 11"
        )

    for split, report in summary["splits"].items():
        if not report["expected_question_count_matches"]:
            raise RuntimeError(
                f"Unexpected CWQ split count: {split}"
            )

    # Verify the dataset has not changed since Notebook 11.
    for filename, expected in summary[
        "configuration"
    ]["source_files"].items():
        path = data_dir / filename
        print("Verifying source:", filename, flush=True)

        if (
            path.stat().st_size != expected["bytes"]
            or file_hash(path) != expected["sha256"]
        ):
            raise RuntimeError(
                "CWQ source changed since Notebook 11: "
                f"{filename}"
            )

    global ENTITIES
    global RELATIONS
    global ENTITY_TO_ID
    global RELATION_TO_ID
    global RELATION_TOKENS

    def read_vocabulary(filename):
        with open(
            data_dir / filename,
            encoding="utf-8",
        ) as file:
            return [
                line.rstrip("\r\n")
                for line in file
            ]

    ENTITIES = read_vocabulary("entities.txt")
    RELATIONS = read_vocabulary("relations.txt")

    ENTITY_TO_ID = {
        value: index
        for index, value in enumerate(ENTITIES)
    }
    RELATION_TO_ID = {
        value: index
        for index, value in enumerate(RELATIONS)
    }
    RELATION_TOKENS = [
        tokens(value)
        for value in RELATIONS
    ]

    if (
        len(ENTITY_TO_ID) != len(ENTITIES)
        or len(RELATION_TO_ID) != len(RELATIONS)
    ):
        raise RuntimeError(
            "Duplicate vocabulary entries"
        )

    splits = manifest["splits"]

    # ----------------------------
    # Split separation and exact-text duplicates
    # ----------------------------

    for first, second in [
        ("train", "dev"),
        ("train", "test"),
        ("dev", "test"),
    ]:
        first_ids = {
            row["qid"] for row in splits[first]
        }
        second_ids = {
            row["qid"] for row in splits[second]
        }

        if first_ids & second_ids:
            raise RuntimeError(
                "Cross-split question ID overlap"
            )

    heldout_texts = {
        row["question_sha256"]
        for split in ["dev", "test"]
        for row in splits[split]
    }

    excluded_train_qids = [
        row["qid"]
        for row in splits["train"]
        if (
            row["eligible"]
            and row["question_sha256"] in heldout_texts
        )
    ]
    excluded_set = set(excluded_train_qids)

    selected = {
        split: [
            row
            for row in splits[split]
            if row["eligible"]
        ]
        for split in ["train", "dev", "test"]
    }

    selected["train"] = [
        row
        for row in selected["train"]
        if row["qid"] not in excluded_set
    ]

    print(
        "\nEligible counts after exact-text training exclusions:",
        {
            split: len(selected_rows)
            for split, selected_rows in selected.items()
        },
    )
    print(
        "Excluded training questions:",
        len(excluded_train_qids),
    )

    pilot_ids = set(pilot["selected_qids"])

    eligible_dev_ids = {
        row["qid"]
        for row in selected["dev"]
    }

    if not pilot_ids or not pilot_ids <= eligible_dev_ids:
        raise RuntimeError(
            "Invalid development pilot selection"
        )

    # ----------------------------
    # Development-only frontier selection
    # ----------------------------

    sweep_configuration = {
        "code_version": CODE_VERSION,
        "audit_signature": summary["signature"],
        "dev_qids": sorted(pilot_ids),
        "limits": FRONTIER_LIMITS,
        "action_cap": ACTION_CAP,
        "recall_tolerance": RECALL_TOLERANCE,
    }
    sweep_signature = fingerprint(sweep_configuration)

    selection_path = (
        output_dir / "cwq_frontier_selection_v1.json"
    )

    selection = (
        read_json(selection_path)
        if selection_path.exists()
        else None
    )

    if (
        selection is None
        or selection.get("signature") != sweep_signature
    ):
        per_question_results = []
        completed = 0

        for _, raw_row in rows(
            data_dir / "dev_simple.json"
        ):
            if question_id(raw_row) not in pilot_ids:
                continue

            example = parse_example(raw_row)

            for limit in [*FRONTIER_LIMITS, None]:
                per_question_results.append(
                    oracle_search(example, limit)
                )

            completed += 1

            if (
                completed % 10 == 0
                or completed == len(pilot_ids)
            ):
                print(
                    "Dev frontier sweep: "
                    f"{completed}/{len(pilot_ids)}",
                    flush=True,
                )

        if completed != len(pilot_ids):
            raise RuntimeError(
                "Missing development pilot questions"
            )

        development_means = {}

        for limit in [*FRONTIER_LIMITS, None]:
            group = [
                result
                for result in per_question_results
                if result["frontier_limit"] == limit
            ]

            key = "full" if limit is None else str(limit)

            development_means[key] = {
                metric: sum(
                    result[metric]
                    for result in group
                ) / len(group)
                for metric in [
                    "recall",
                    "oracle_recognition_f1",
                    "exact",
                    "actions",
                    "retrieved_edges",
                ]
            }

        # Reproduce the two existing Notebook-11 operating points.
        for new_key, old_key in [
            ("16", "oracle_b24"),
            ("full", "oracle_full_frontier_b24"),
        ]:
            for metric in [
                "recall",
                "oracle_recognition_f1",
                "exact",
                "actions",
                "retrieved_edges",
            ]:
                difference = abs(
                    development_means[new_key][metric]
                    - pilot["results"][old_key][metric]
                )

                if difference > 1e-9:
                    raise RuntimeError(
                        "Notebook-11 regression mismatch: "
                        f"{new_key}, {metric}"
                    )

        target_recall = (
            development_means["full"]["recall"]
            - RECALL_TOLERANCE
        )

        qualified_limits = [
            limit
            for limit in FRONTIER_LIMITS
            if (
                development_means[str(limit)]["recall"]
                >= target_recall
            )
        ]

        if qualified_limits:
            chosen_limit = min(qualified_limits)
        else:
            # Transparent fallback if no tested limit reaches the target.
            chosen_limit = max(
                FRONTIER_LIMITS,
                key=lambda limit: (
                    development_means[str(limit)]["recall"],
                    -limit,
                ),
            )

        selection = {
            "signature": sweep_signature,
            "configuration": sweep_configuration,
            "selected_frontier_limit": chosen_limit,
            "target_recall": target_recall,
            "target_met": bool(qualified_limits),
            "development_means": development_means,
            "selection_rule": (
                "Smallest limit within 0.02 recall of full frontier; "
                "otherwise best recall, smaller-limit tie break."
            ),
            "notebook11_regression_passed": True,
            "oracle_recognition": True,
            "greedy_oracle_not_optimal": True,
            "per_question": per_question_results,
        }

        save_json(selection_path, selection)

    frontier_limit = int(
        selection["selected_frontier_limit"]
    )

    print("\nDevelopment frontier results:")
    print(
        json.dumps(
            selection["development_means"],
            indent=2,
        )
    )
    print("Selected frontier limit:", frontier_limit)
    print("Recall target met:", selection["target_met"])

    if not selection["target_met"]:
        print(
            "The selected finite shortlist remains below the "
            "development target. This is recorded for review."
        )

    # ----------------------------
    # Save the cleaned experiment manifest
    # ----------------------------

    configuration = {
        "code_version": CODE_VERSION,
        "audit_signature": summary["signature"],
        "seed": SEED,
        "max_hops": MAX_HOPS,
        "frontier_limit": frontier_limit,
        "action_cap": ACTION_CAP,
        "snapshot_steps": SNAPSHOT_STEPS,
        "train_candidates_per_state": TRAIN_CANDIDATES_PER_STATE,
        "dev_candidates": "all shortlisted",
        "train_storage_sampling": "uniform, label-independent",
        "policies": ["oracle", "relation", "random"],
        "chunk_questions": CHUNK_QUESTIONS,
        "selected_qids": {
            split: [
                row["qid"]
                for row in selected_rows
            ]
            for split, selected_rows in selected.items()
        },
        "excluded_train_qids": excluded_train_qids,
        "controller_text_version": "webqsp_notebook06_v1",
    }

    signature = fingerprint(configuration)

    manifest_path = (
        processed_dir / "cwq_manifest_v2_full.json"
    )

    save_json(
        manifest_path,
        {
            "signature": signature,
            "configuration": configuration,
            "splits": selected,
            "excluded_train_qids": excluded_train_qids,
            "test_state_generation": False,
            "test_search_evaluation": False,
        },
    )

    # ----------------------------
    # Generate train/dev states
    # ----------------------------

    generated = {}

    for split in ["train", "dev"]:
        generated[split] = build_split(
            split,
            selected[split],
            data_dir,
            output_dir,
            processed_dir,
            signature,
            frontier_limit,
        )

    # ----------------------------
    # Save final reports
    # ----------------------------

    integrity_path = (
        output_dir / "cwq_state_integrity_audit_v1.json"
    )

    save_json(
        integrity_path,
        {
            "signature": signature,
            "splits": generated,
            "passed": True,
            "full_frontier_stop_labels": True,
            "train_candidates_sampled_without_labels": True,
            "all_dev_shortlist_candidates_retained": True,
            "input_audit": (
                "Exact reconstruction from approved observed features; "
                "no substring-based gold-word checks."
            ),
            "note": (
                "STOP labels refer to the full frontier. "
                "Do not infer STOP from sampled training candidates."
            ),
        },
    )

    generation_path = (
        output_dir / "cwq_state_generation_summary_v1.json"
    )

    save_json(
        generation_path,
        {
            "signature": signature,
            "configuration": configuration,
            "splits": generated,
            "minutes": (time.time() - started) / 60,
            "frontier_selection": str(selection_path),
            "frontier_target_met": selection["target_met"],
            "manifest_path": str(manifest_path),
            "answer_recognition_during_state_generation": (
                "oracle; learned recognition required "
                "for end-to-end evaluation"
            ),
            "test_questions": len(selected["test"]),
            "test_states_generated": False,
            "notes": [
                (
                    "Read ONLY candidate.model_text as controller input; "
                    "gain and STOP fields are targets/diagnostics."
                ),
                (
                    "Global STOP and shortlist STOP differ when "
                    "candidate retrieval hides useful branches."
                ),
                (
                    "Dev thresholds must be calibrated for CWQ separately; "
                    "WebQSP thresholds are unchanged."
                ),
                (
                    "This is a conditional multi-answer subset, "
                    "not the official full-CWQ benchmark."
                ),
                (
                    "Explicit text-field checks do not establish "
                    "absence of every possible leakage source."
                ),
            ],
        },
    )

    artifact_paths = [
        selection_path,
        generation_path,
        integrity_path,
        manifest_path,
        *[
            Path(report["path"])
            for report in generated.values()
        ],
    ]

    inventory_path = (
        output_dir / "artifact_inventory_v1.json"
    )

    artifact_entries = []

    for path in artifact_paths:
        known_hash = next(
            (
                report["sha256"]
                for report in generated.values()
                if report["path"] == str(path)
            ),
            None,
        )

        artifact_entries.append({
            "name": path.name,
            "path": str(path),
            "bytes": path.stat().st_size,
            "sha256": known_hash or file_hash(path),
        })

    save_json(
        inventory_path,
        {
            "signature": signature,
            "files": artifact_entries,
        },
    )

    print("\n" + "=" * 90)
    print("NOTEBOOK 12 COMPLETED")
    print("=" * 90)
    print("CPU preprocessing only.")
    print("WebQSP artifacts remain unchanged.")
    print("No test search evaluation was performed.")

    print("\nGenerated training files:")
    for report in generated.values():
        print(report["path"])

    print("\nSEND THESE FOUR REPORTS:")
    for path in [
        selection_path,
        generation_path,
        integrity_path,
        inventory_path,
    ]:
        print(path)

    print(
        "\nReview these reports before starting "
        "CWQ controller training."
    )
    print("=" * 90)


main()

Mounted at /content/drive
NOTEBOOK 12: CWQ FRONTIER SELECTION AND STATE GENERATION
Runtime: CPU
No downloads, model training, or test search evaluation.
Output directory: /content/drive/MyDrive/SetStop_KGQA/outputs/cwq_preprocessing_v1
Action cap: 24
Frontier limits: [16, 32, 64, 128]
Verifying source: train_simple.json
Verifying source: dev_simple.json
Verifying source: test_simple.json
Verifying source: entities.txt
Verifying source: relations.txt

Eligible counts after exact-text training exclusions: {'train': 4644, 'dev': 520, 'test': 495}
Excluded training questions: 1
Dev frontier sweep: 10/50
Dev frontier sweep: 20/50
Dev frontier sweep: 30/50
Dev frontier sweep: 40/50
Dev frontier sweep: 50/50

Development frontier results:
{
  "16": {
    "recall": 0.5716666666666667,
    "oracle_recognition_f1": 0.578,
    "exact": 0.56,
    "actions": 9.84,
    "retrieved_edges": 76.0
  },
  "32": {
    "recall": 0.66,
    "oracle_recognition_f1": 0.6666666666666667,
    "exact": 0.64,
    "